# 01 · 数据准备（Colab）

下载语料、建清单、按平稳性给噪声分类、合成固定测试集。

## 数据设计

| 用途 | 数据集 | 规模 | 为什么是它 |
|---|---|---|---|
| **训练干净语音** | DNS Challenge 5 `read_speech` | 1 切片 ≈ 19 小时 | 学术界公认基准；切片可**部分解压**做小规模子集（见下） |
| **训练噪声** | DNS `noise_fullband`（AudioSet + Freesound） | 3 分片 ≈ 14.2 GB | 真实录制，覆盖面广；按平稳性自动分成两组 |
| **房间冲激响应** | DNS `impulse_responses` + 本项目合成 | 0.26 GB + 0 | 真实的验证泛化，合成的做**受控 RT60 扫描** |
| **中文 ASR 评测** | WenetSpeech `test_meeting` | 220 MB parquet | 真实会议录音，带中文转写 |

## 核心设计：训练用英文，评测用中文

这不是凑合，是**刻意的**。前端增强模型在 DNS（英文为主）上训练，
却直接在中文 WenetSpeech 上测 CER —— 如果 CER 仍然改善，就证明模型学到的是
**纯粹的声学/频域去噪规律**，而不是过拟合到某个语言的发音模式。
同语种自训自测拿不到这个论证。

## 为什么噪声要区分稳态 / 非稳态

这是"传统 DSP vs 神经网络"这条对照的核心抓手：

- **稳态**（风扇、空调、白噪声）：谱统计平稳，MCRA 类噪声估计跟得上，
  谱减/维纳/MMSE-LSA 表现好且算力极低；
- **非稳态**（键盘、关门、餐具、babble）：统计特性瞬变，传统方法跟不上噪声谱，
  失效或产生明显音乐噪声 —— 这正是神经网络该赢的地方。

DNS 的噪声文件**不带平稳性标注**，所以用 `rtse.dsp.stationarity` 从信号本身算
（去趋势帧能量动态范围），本地已在 7 类已知噪声上验证过分类正确。

## 为什么 RIR 要合成 + 真实两种都用

- **合成 RIR**（镜像源法）：参数完全可控，能精确指定 RT60，
  做 0.2 / 0.4 / 0.6 / 0.8 s 的**严格量化扫描**；
- **真实 RIR**（DNS 实测）：含非均匀漫反射、墙面材质不对称吸收、麦克风频响失真，
  检验模型在真实声学环境下的泛化，避免"仿真过拟合"。

两者在测试集里**并行分层**，可以直接对比"合成 RIR 上的成绩比真实 RIR 好多少"。

## 执行前

1. 先跑「配置」cell，确认 `DRIVE_ROOT` 与你的实际目录一致
2. `rtse-colab.zip` 必须已上传到 `DRIVE_ROOT`（本地 `uv run python scripts/pack_for_colab.py` 生成）
3. **第一次建议先把 `QUICK_TEST = True`** 跑一遍（只下 WenetSpeech 约 520 MB，验证链路）

In [1]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS5 干净语音的 split 切片数。每片 5.24 GB，实测约 **19 小时**，
# 落在"20~30 小时可管理子集"这个目标区间内。
# 切片档解压到末尾会报 EOF，属正常（详见 fetch_dns 的说明）。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# True = 跳过 DNS 大文件，只下 WenetSpeech（约 520 MB）验证整条链路。
QUICK_TEST = False

# ── 小规模跑通模式 ─────────────────────────────────────────────────────
# True = 大幅缩小**用量**与**训练时长**，验证「数据→训练→导出→评测」整条链路。
# 缩的不是下载量（DNS 分片是最小单位，该下多少还是多少），而是"用多少条"和"跑多少轮"：
#   测试集   50 格 × 3 = 150 条（正式 15/格 = 750）
#   噪声分类 抽 400 条做平稳性判决（正式 4000）
#   训练     2000 样本/epoch × 3 epoch，只跑 crn-nano（正式 20000 × 60，两档）
# 跑通之后改成 False 再跑正式版。
SMOKE_RUN = True

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 英文训练 + WenetSpeech 中文评测)"}')
print(f'  规模      {"⚡ 小规模跑通（SMOKE_RUN=True，结果不作数）" if SMOKE_RUN else "正式规模"}')
print()

!df -h /content | tail -1

目录布局
──────────────────────────────────────────────────────────────────────────
  代码包(需手动上传)  /content/drive/MyDrive/Audio AI/RTSE/rtse-colab.zip
  压缩包缓存          /content/drive/MyDrive/Audio AI/RTSE/archives
  语料解压目标        /content/rtse_work/data
  数据清单            /content/drive/MyDrive/Audio AI/RTSE/manifest.json
  固定测试集          /content/drive/MyDrive/Audio AI/RTSE/testset
  训练断点  ★         /content/drive/MyDrive/Audio AI/RTSE/checkpoints/<模型名>/{last,best}.pt
  导出模型  ★         /content/drive/MyDrive/Audio AI/RTSE/models/<模型名>.onnx
──────────────────────────────────────────────────────────────────────────
  数据模式  hybrid
  语料      完整(DNS 英文训练 + WenetSpeech 中文评测)

overlay         236G   48G  189G  20% /


In [3]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
# **重新导入前必须把已加载的 rtse 从 sys.modules 里清掉。**
# 上面的 unzip 换的是磁盘上的文件，而 `import rtse` 对**已经导入过**的模块是空操作：
# 同一个 Colab 会话里重跑本 cell，磁盘上是新代码、内存里跑的还是旧的。
# 这个症状极具迷惑性——报错的行号来自旧文件，跟你手里的新文件对不上号，
# 会让人以为"包没传上去"而反复重传（见 docs/ISSUES.md I-13 / I-28）。
for _m in [m for m in list(sys.modules) if m == 'rtse' or m.startswith('rtse.')]:
    del sys.modules[_m]
import rtse
# 自证：把**实际加载的文件路径和改动时间**打出来。
# "我改的代码到底有没有在跑"必须是可观测的事实，不能靠推断（I-28）。
print('已加载 rtse ←', rtse.__file__)
print('           改动时间',
      time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(rtse.__file__))),
      '| 若这个时间不是你刚打包的时刻，说明跑的还是旧代码')
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.4 MB/s eta 0:00:00
已加载 rtse ← /content/rtse_work/rtse-src/src/rtse/__init__.py
           改动时间 08-08 20:11 | 若这个时间不是你刚打包的时刻，说明跑的还是旧代码
rtse 0.1.0 | SR 16000 | n_fft 512 | hop 256
torch 2.11.0+cu128 | CUDA True | Tesla T4


In [4]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。Colab 上的 STFT 与本地哪怕差一点，训练出来的模型拿回本地就会
# 掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

COLA 偏差          : 4.44e-16
numpy 完美重构      : 1.33e-15
torch/numpy STFT   : 1.22e-07 (相对)
torch 往返重构      : 7.15e-07
dBFS 标定(满幅正弦) : 0.001 dB  (应为 0.000)

✅ Colab 与本地是同一条链路。


## 1. 下载 DNS Challenge 语料

分片是**独立的** `.tar.bz2`，可以只下需要的几片 —— 这是能做小规模子集的前提。
（DNS5 的 clean speech 是 `split` 切片，必须全部下载才能拼接解压，用不了。）

下载支持断点续传（`wget -c`）；已下好的会跳过，Colab 断线重连后重跑不会从头再来。

In [5]:
DNS_BASE = 'https://dnschallengepublic.blob.core.windows.net/dns5archive/V5_training_dataset'

# lbzip2：多线程解压 bzip2。噪声分片是 .tar.bz2，单线程解 5 GB 要 ~9 分钟，
# 而 Colab 每个会话都得重解一遍（/content 会被清空），这是最大的一块固定开销。
# 装不上也能跑，只是退回单线程。
!apt-get -qq install -y lbzip2 > /dev/null 2>&1 || true
print('lbzip2:', '可用（解压将并行）' if shutil.which('lbzip2') else '不可用（退回单线程 bzip2）')

def fetch_dns(name, blob_path, expect_min_wavs=50, partial_ok=False):
    """下载 → 解压一个 DNS 分片。带下载/解压双标记，支持断点续传与跨会话复用。

    Args:
        partial_ok: 该分片是 `split` 切片（干净语音），解压到末尾必然报
            "Unexpected EOF"。设 True 时忽略这个错误——只要解出足够多的
            完整文件就算成功。校验靠**实际解出的 wav 数量**，不靠 tar 的返回码。

    校验方式统一是"解压后递归扫到的 wav 数量"，而不是断言某个具体子目录名——
    实测 DNS5 语音解出来的路径是 `mnt/dnsv5/clean/read_speech/...`，
    嵌套好几层且没写在官方文档里。下游 scan() 本来就是递归扫描，不关心层数。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    # 解压标记里记下**当时解出的文件数**和格式版本。只有"新格式 + 文件数对得上"
    # 才认为可信。旧格式（空文件 touch 出来的）可能是 I-26 修复前写下的——
    # 那时截断的解压也会被当成成功（实测有分片只解出 1156/7739 却被记成已完成），
    # 一律重解一遍。这样**修复能追溯地清理掉此前留下的坏状态**，
    # 而不是只对将来生效、让已经写坏的标记永远把分片挡在门外。
    if os.path.exists(ex_mark):
        try:
            mark = json.loads(Path(ex_mark).read_text())
        except Exception:
            mark = None
        n_now = count_wavs()
        if isinstance(mark, dict) and mark.get('v', 0) >= 2 and mark.get('n') == n_now:
            print(f'[skip  ] {name} 已解压（{n_now} 个 wav）'); return True
        why = ('标记是旧格式，无法确认当时是否解压完整'
               if mark is None else
               f'文件数与标记不符（记录 {mark.get("n")}，实际 {n_now}）')
        print(f'[recheck] {name} {why} —— 重解一遍')

    def free_gb(path):
        try:
            return shutil.disk_usage(path).free / 1e9
        except Exception:
            return float('nan')

    # 下载标记里记下**当时的字节数**，缓存命中时核对一遍。
    # 只看"标记在不在"是不够的：压缩包可能被截断、被覆盖、或下到一半留下残file。
    # 旧格式（`touch` 出来的空文件）没有这个信息，就地升级成新格式，
    # **不重新下载**——那是 20 GB，代价太大，而且现有包已逐个核对过与服务器一致。
    cached_ok = False
    if os.path.exists(dl_mark) and os.path.exists(archive):
        size_now = os.path.getsize(archive)
        try:
            rec = json.loads(Path(dl_mark).read_text())
        except Exception:
            rec = None
        if isinstance(rec, dict) and rec.get('bytes') is not None:
            if rec['bytes'] == size_now:
                cached_ok = True
                print(f'[cached] {name} 压缩包已在 Drive（{size_now/1e9:.2f} GB，大小与记录一致）')
            else:
                # 用**字节数**报差异，不用 GB：两个只差几 MB 的数字，
                # 格式化成 GB 会显示成一模一样，等于没给信息。
                print(f'[FAIL  ] {name} 压缩包大小与记录不符 —— 文件被改动或截断。')
                print(f'         记录 {rec["bytes"]:,} 字节，实际 {size_now:,} 字节'
                      f'（差 {size_now - rec["bytes"]:+,}）')
                print(f'         删掉 {archive} 和 {dl_mark} 后重跑，或直接重跑让 -c 续传。')
                return False
        else:
            cached_ok = True
            Path(dl_mark).write_text(json.dumps({'v': 2, 'bytes': size_now}))
            print(f'[cached] {name} 压缩包已在 Drive（{size_now/1e9:.2f} GB，标记已补记大小）')
    if not cached_ok:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        wlog = f'{ARCHIVE_DIR}/.{name}.wget.log'
        # 进度**必须可见**：几个 GB 的下载如果一声不吭，中途看起来和卡死没区别。
        # 同时又要留下日志，失败时才说得出原因，所以用 tee 兼顾两者。
        # `-T 60` 的超时在长连接上很容易触发，但 `-c` 会断点续传——
        # 重跑本 cell 就能接着下，**不要删掉已下载的部分**。
        # 走 bash 是因为要取 PIPESTATUS（Colab 的 /bin/sh 是 dash，不支持）。
        cmd = (f'wget --progress=dot:giga -c -T 60 -O {shq(archive)} {shq(url)} '
               f'2>&1 | tee {shq(wlog)}; exit ${{PIPESTATUS[0]}}')
        rc = subprocess.run(['bash', '-c', cmd]).returncode
        size = os.path.getsize(archive) if os.path.exists(archive) else 0
        if rc != 0 or size < 1e6:
            # **不要直接断言"blob 路径变了"**——那只是众多可能之一，而且是最不可能的那个。
            # 实测最常见的是**空间不足**：hybrid 模式下归档写在 Drive 上，
            # 本批归档合计约 20 GB，而 Drive 免费版只有 15 GB。
            # 先把证据摆出来（wget 原话 + 两个卷的剩余空间），再让人去判断。
            print(f'[FAIL  ] {name} 下载未完成（wget 退出码 {rc}，已落盘 {size/1e9:.2f} GB）')
            print('         ⚠️ 多数情况下这只是超时中断，**已下载的部分是有效的**：'
                  '直接重跑本 cell，-c 会接着下。')
            tail = subprocess.run(f'tail -n 3 {shq(wlog)}', shell=True,
                                  capture_output=True, text=True).stdout.strip()
            if tail:
                print('         wget: ' + tail.replace(chr(10), chr(10) + '         '))
            print(f'         剩余空间：归档卷 {free_gb(ARCHIVE_DIR):.1f} GB，'
                  f'解压卷 {free_gb(DATA):.1f} GB')
            print('         本批归档合计约需 20 GB（语音 5.2 + 噪声 14.2 + IR 0.3）。')
            print('         排查顺序：① 上面两个卷的剩余空间够不够；'
                  '② Drive 挂载是否还活着（ls 一下 DRIVE 目录）；')
            print('         ③ 都正常再去 https://github.com/microsoft/DNS-Challenge '
                  '核对 blob 路径（这一项本地 HEAD 实测过 200，最不可能）。')
            return False
        Path(dl_mark).write_text(json.dumps({'v': 2, 'bytes': size}))

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    # 压缩格式**按文件头判断，不能看扩展名**：切片档叫 `read_speech.tgz.partaa`，
    # 结尾是 `.partaa` 而不是 `.tgz`，用 endswith 会判成 bzip2；GNU tar 拿 -j 去解
    # gzip 流会直接报 "is not a bzip2 file" 且一个文件都不解出。
    # Colab 上实际踩过：语音分片解出 0 个 wav（见 docs/ISSUES.md I-25）。
    with open(archive, 'rb') as fh:
        magic = fh.read(2)
    if magic == b'\x1f\x8b':
        flag, prog = 'xzf', None
    else:
        # **bzip2 解压是单线程 CPU 瓶颈**：5 GB 分片实测约 9 分钟，全程只吃一个核。
        # lbzip2 能对**任意** bzip2 流做多线程解压（pbzip2 只能并行它自己压出来的），
        # Colab 两个 vCPU 大致能快一倍。装不上就退回单线程，不影响正确性。
        prog = 'lbzip2' if shutil.which('lbzip2') else None
        flag = 'xf' if prog else 'xjf'
    decomp = f'--use-compress-program={prog} ' if prog else ''
    # 切片档忽略 tar 的非零返回码（末尾必然 EOF），靠文件数判断成败。
    # stderr 落盘而不是丢进 /dev/null——失败时要能说出为什么失败。
    log = f'{out_dir}/.untar.log'
    with open(log, 'w') as lf:
        rc_tar = subprocess.run(f'tar {decomp}-{flag} {shq(archive)} -C {shq(out_dir)}',
                                shell=True, stderr=lf).returncode
    n = count_wavs()

    def tar_err():
        t = subprocess.run(f'tail -n 5 {shq(log)}', shell=True,
                           capture_output=True, text=True).stdout.strip()
        return ('\n         tar: ' + t.replace(chr(10), chr(10) + '         ')) if t else ''

    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav（用 {flag} 解，文件头 {magic!r}）。' + tar_err())
        print(f'         剩余空间：解压卷 {free_gb(DATA):.1f} GB')
        print(f'         删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False
    # 切片档解到末尾必然 EOF，退出码非零属正常；**其余压缩包退出码非零意味着解压被截断**，
    # 而此时 wav 数很可能仍然过线，于是被当成成功放过去。
    # 实测踩过：噪声分片只解出 1156/7739 个文件却报 [done]，训练数据悄悄少了 85%。
    if rc_tar != 0 and not partial_ok:
        print(f'[FAIL  ] {name} 解压未完成：tar 退出码 {rc_tar}，只解出 {n} 个 wav。' + tar_err())
        print(f'         剩余空间：解压卷 {free_gb(DATA):.1f} GB —— 空间不足是最常见原因。')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive); Path(dl_mark).unlink(missing_ok=True)
    # 记下文件数，下次才能判断"这份解压结果还是不是完整的那一份"
    Path(ex_mark).write_text(json.dumps({'v': 2, 'n': n}))
    note = '（切片档，尾部 EOF 属正常）' if partial_ok else ''
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒 {note}')
    return True


def dns_shards():
    """按配置生成要下载的分片清单 (名字, blob 路径)。"""
    # 语音是 split 切片，按字母序 partaa/partab/...；每片约 5.24 GB ≈ 19 小时。
    # ⚠️ **只有 partaa 能单独解压**：gzip 头只在第一片里，partab 及之后都是
    # 裸的流中段，单独拿去 tar 解必然失败（文件头实测是 b'\n\xc5' 这类随机字节）。
    # 要更多语音只能把所有片下全再 `cat *.part* | tar xz`，那是 110 GB。
    assert N_SPEECH_SHARDS == 1, (
        'N_SPEECH_SHARDS 只能是 1：DNS5 语音是 split 切片，gzip 头只在 partaa 里，'
        '后续切片无法独立解压。需要更多数据请改用别的数据源，或准备 110 GB 下全部切片。')
    parts = ['aa', 'ab', 'ac', 'ad', 'ae']
    sp = [(f'dns_speech_{p}', f'Track1_Headset/read_speech.tgz.part{p}')
          for p in parts[:N_SPEECH_SHARDS]]
    nz = [(f'dns_noise_audioset_{i:03d}',
           f'noise_fullband/datasets_fullband.noise_fullband.audioset_{i:03d}.tar.bz2')
          for i in range(N_AUDIOSET_SHARDS)]
    nz += [(f'dns_noise_freesound_{i:03d}',
            f'noise_fullband/datasets_fullband.noise_fullband.freesound_{i:03d}.tar.bz2')
           for i in range(N_FREESOUND_SHARDS)]
    # ⚠️ IR 分片在 blob 根目录下，**没有** `impulse_responses/` 前缀
    # （语音和噪声分片才有目录前缀）。写错会 404 —— 本地 HEAD 请求实测确认过。
    ir = [('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2')]
    return sp, nz, ir

sp_shards, nz_shards, ir_shards = dns_shards()
print(f'计划下载：语音 {len(sp_shards)} 片、噪声 {len(nz_shards)} 片、IR {len(ir_shards)} 片')
free = shutil.disk_usage(DATA).free / 1e9
print(f'解压卷可用 {free:.1f} GB（估计需要 40~60 GB）')

if QUICK_TEST:
    print('\n[QUICK_TEST] 跳过 DNS 下载，只用 WenetSpeech 验证链路')
    results = {}
else:
    t0 = time.time()
    results = {}
    for n, b in sp_shards: results[n] = fetch_dns(n, b, expect_min_wavs=500, partial_ok=True)
    for n, b in nz_shards: results[n] = fetch_dns(n, b, expect_min_wavs=100)
    for n, b in ir_shards: results[n] = fetch_dns(n, b, expect_min_wavs=50)
    print(f'\n总耗时 {(time.time()-t0)/60:.1f} 分钟   结果: {results}')
    assert all(results.values()), '有分片没就绪，看上面的 FAIL 信息'

!df -h {shq(DATA)} | tail -1

lbzip2: 可用（解压将并行）
计划下载：语音 1 片、噪声 3 片、IR 1 片
解压卷可用 202.3 GB（估计需要 40~60 GB）
[cached] dns_speech_aa 压缩包已在 Drive（5.24 GB，标记已补记大小）
[unpack] dns_speech_aa  (5.24 GB) → /content/rtse_work/data/dns_speech_aa
[done  ] dns_speech_aa   14279 个 wav   解压耗时 147 秒 （切片档，尾部 EOF 属正常）
[cached] dns_noise_audioset_000 压缩包已在 Drive（5.36 GB，标记已补记大小）
[unpack] dns_noise_audioset_000  (5.36 GB) → /content/rtse_work/data/dns_noise_audioset_000
[done  ] dns_noise_audioset_000   8000 个 wav   解压耗时 350 秒 
[cached] dns_noise_audioset_001 压缩包已在 Drive（5.36 GB，标记已补记大小）
[unpack] dns_noise_audioset_001  (5.36 GB) → /content/rtse_work/data/dns_noise_audioset_001
[done  ] dns_noise_audioset_001   8000 个 wav   解压耗时 355 秒 
[cached] dns_noise_freesound_000 压缩包已在 Drive（3.47 GB，标记已补记大小）
[unpack] dns_noise_freesound_000  (3.47 GB) → /content/rtse_work/data/dns_noise_freesound_000
[done  ] dns_noise_freesound_000   8000 个 wav   解压耗时 267 秒 
[cached] dns_ir 压缩包已在 Drive（0.26 GB，标记已补记大小）
[unpack] dns_ir  (0.26 GB) → /content/rtse_work/

## 2. 下载 WenetSpeech 中文评测集

**为什么用 HuggingFace 镜像而不是官方渠道**：WenetSpeech 官方
（wenet.org.cn）需要填 Google 表单拿密码，没法在 notebook 里脚本化。
`lmms-lab/WenetSpeech` 这个镜像不需要登录也不需要 token，
`test_meeting` 的音频和中文转写都打包在一个 220 MB 的 parquet 里。

> ⚠️ 该镜像的 `test_net` 只有 124 个文件（残缺），**不要用**。
> 完整可用的是 `dev`（13825 条）和 `test_meeting`（8370 条）。
> 这里用 `test_meeting` —— 真实会议录音，对"远场"这个主题比朗读语料更贴题。

In [6]:
import pandas as pd

WNS_URL = ('https://huggingface.co/datasets/lmms-lab/WenetSpeech/resolve/main/'
           'data/test_meeting-00000-of-00001.parquet')
wns_parquet = f'{ARCHIVE_DIR}/wenetspeech_test_meeting.parquet'

if not os.path.exists(wns_parquet):
    print(f'[get   ] WenetSpeech test_meeting ← {WNS_URL}')
    rc = os.system(f'wget -q --show-progress -c -T 60 -O {shq(wns_parquet)} {shq(WNS_URL)}')
    assert rc == 0 and os.path.getsize(wns_parquet) > 1e6, '下载失败，检查网络或镜像是否还在'
else:
    print(f'[cached] WenetSpeech parquet 已存在 ({os.path.getsize(wns_parquet)/1e6:.0f} MB)')

wns = pd.read_parquet(wns_parquet)
print(f'\n共 {len(wns)} 条')
print('列：', list(wns.columns))
print('\n第一条样例：')
row0 = wns.iloc[0]
for c in wns.columns:
    v = row0[c]
    print(f'  {c}: {str(v)[:80] if not isinstance(v, (dict, bytes)) else type(v).__name__}')

[cached] WenetSpeech parquet 已存在 (220 MB)

共 8370 条
列： ['utt_id', 'audio', 'text', 'begin_time', 'end_time', 'aid', 'audio_path']

第一条样例：
  utt_id: TEST_MEETING_T0000000000_S00000
  audio: dict
  text: 咱们的第八层是属于租的人家的
  begin_time: 0.0
  end_time: 1.94
  aid: TEST_MEETING_T0000000000
  audio_path: WenetSpeechDataset/audio/test_meeting/third_party/B00000/TEST_MEETING_T000000000


## 3. 建立文件清单 + 噪声按平稳性分类

语音按**说话人**划分（避免同一人同时出现在训练和测试，模型靠记音色作弊）。
噪声用 `rtse.dsp.stationarity` 逐个算平稳性并分成两组 —— 这是后面
"DSP 在稳态噪声上够用、在非稳态上失效"这条对照能不能立住的前提。

In [7]:
import random
from rtse.dsp.stationarity import stationarity_features, DEFAULT_DR_THRESHOLD_DB
from rtse.audio.io import read_audio
from tqdm.auto import tqdm

def scan(root, exts=('.wav', '.flac')):
    root = Path(root)
    if not root.exists(): return []
    return sorted(str(p) for p in root.rglob('*') if p.suffix.lower() in exts)

speech_all, noise_all, rir_all = [], [], []
if not QUICK_TEST:
    for n, _ in sp_shards: speech_all += scan(f'{DATA}/{n}')
    for n, _ in nz_shards: noise_all += scan(f'{DATA}/{n}')
    for n, _ in ir_shards: rir_all += scan(f'{DATA}/{n}')

print(f'语音 {len(speech_all):>7} 条')
print(f'噪声 {len(noise_all):>7} 条')
print(f'RIR  {len(rir_all):>7} 条')
if not QUICK_TEST:
    assert speech_all and noise_all and rir_all, '有类别扫不到文件，检查上一步解压结果'

语音   14279 条
噪声   24000 条
RIR    60248 条


In [8]:
# ── 噪声平稳性分类 ─────────────────────────────────────────────────────
# 全量算太慢（几万个文件），抽样一部分做分类；每个文件只读前 10 秒就够判断。
MAX_NOISE_TO_CLASSIFY = 400 if SMOKE_RUN else 4000
rnd = random.Random(42)
noise_pool = noise_all[:]
rnd.shuffle(noise_pool)
noise_pool = noise_pool[:MAX_NOISE_TO_CLASSIFY]

stationary, nonstationary, skipped = [], [], 0
for p in tqdm(noise_pool, desc='噪声平稳性分类'):
    try:
        y = read_audio(p)[:16000 * 10]
    except Exception:
        skipped += 1; continue
    y_ac = y - y.mean()
    if not np.all(np.isfinite(y)) or np.mean(y_ac ** 2) <= 1e-12:
        # 常量/静音文件没有可定义的 SNR；以前会混进 stationary 组，最终生成 inf。
        skipped += 1; continue
    f = stationarity_features(y_ac)
    if f.n_frames < 32:           # 太短，无法可靠判断
        skipped += 1; continue
    (stationary if f.detrended_dynamic_range_db < DEFAULT_DR_THRESHOLD_DB
     else nonstationary).append(p)

print(f'\n稳态   {len(stationary):>5} 条')
print(f'非稳态 {len(nonstationary):>5} 条')
print(f'跳过   {skipped:>5} 条（太短或读取失败）')
print(f'\n门限 {DEFAULT_DR_THRESHOLD_DB} dB —— 这个值是在合成噪声上标定的，'
      f'真实录音分布更连续，如果两组比例悬殊（比如 9:1）就该调它。')
assert stationary and nonstationary, '有一组是空的，门限需要重新标定'

噪声平稳性分类:   0%|          | 0/4000 [00:00<?, ?it/s]


稳态    2064 条
非稳态  1926 条
跳过      10 条（太短或读取失败）

门限 9.0 dB —— 这个值是在合成噪声上标定的，真实录音分布更连续，如果两组比例悬殊（比如 9:1）就该调它。


In [9]:
# ── 划分 train/test ────────────────────────────────────────────────────
# 语音按**说话人**划分，不是按文件随机划分：按文件随机会让同一个人同时出现在
# 训练集和验证集里，模型能靠"记住这个人的音色"作弊，指标虚高。
#
# ⚠️ DNS5 的文件名形如 `book_00000_chp_0009_reader_06709_0_seg_1_seg1.wav`，
# 说话人 id 藏在 `reader_XXXXX` 这一段里。**不能用 `stem.split('_')[0]`**——
# 那会对每个文件都返回 "book"，所有语音归成一个说话人，划分彻底失效。
# 这个 bug 本地用 Range 请求取前 80 MB 实测抓到过（231 个文件切出 47 位说话人）。
import re

def speaker_of(p):
    m = re.search(r'reader_(\d+)', str(p))
    return m.group(1) if m else Path(p).stem

spk_probe = sorted({speaker_of(p) for p in speech_all[:2000]})
print(f'说话人 id 提取自检：前 2000 个文件切出 {len(spk_probe)} 位说话人，'
      f'样例 {spk_probe[:5]}')
assert 1 < len(spk_probe) < len(speech_all[:2000]) * 0.9, (
    '说话人提取异常：要么全归成一个人（正则没匹配上），要么几乎每个文件一个人'
    '（文件名格式变了）。两种情况都会让划分失去意义，必须先修这里。'
)

spk = sorted({speaker_of(p) for p in speech_all})
random.Random(20260807).shuffle(spk)
n_val = max(2, len(spk) // 10)
spk_val = set(spk[:n_val])
split = {'train': [], 'val': []}
for p in speech_all:
    split['val' if speaker_of(p) in spk_val else 'train'].append(p)
print(f'说话人 {len(spk)} 位 → train {len(spk)-n_val} / val {n_val}')
for k, v in split.items():
    print(f'  语音 {k:>5}: {len(v):>7} 条')

# 噪声与 RIR 也划分：测试用的必须是训练没见过的
def split_list(xs, frac=0.2, seed=42):
    xs = xs[:]; random.Random(seed).shuffle(xs)
    cut = max(1, int(len(xs) * frac))
    return xs[cut:], xs[:cut]           # (train, test)

st_train, st_test = split_list(stationary)
ns_train, ns_test = split_list(nonstationary)
rir_train, rir_test = split_list(rir_all)
print(f'  稳态噪声  train/test: {len(st_train)}/{len(st_test)}')
print(f'  非稳态噪声 train/test: {len(ns_train)}/{len(ns_test)}')
print(f'  真实 RIR  train/test: {len(rir_train)}/{len(rir_test)}')

manifest = {
    'version': 'dns_wenetspeech',
    'quick_test': QUICK_TEST,
    'data_dir': DATA,
    'speech': split,
    'noise_train': st_train + ns_train,
    'noise_test': st_test + ns_test,
    'noise_stationary_train': st_train, 'noise_stationary_test': st_test,
    'noise_nonstationary_train': ns_train, 'noise_nonstationary_test': ns_test,
    'rir_train': rir_train, 'rir_test': rir_test,
    'stationarity_threshold_db': DEFAULT_DR_THRESHOLD_DB,
}
Path(f'{DRIVE}/manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
print(f'\n清单已写入 {DRIVE}/manifest.json')

说话人 id 提取自检：前 2000 个文件切出 35 位说话人，样例 ['00732', '00927', '01062', '01326', '01593']
说话人 318 位 → train 287 / val 31
  语音 train:   12975 条
  语音   val:    1304 条
  稳态噪声  train/test: 1652/412
  非稳态噪声 train/test: 1541/385
  真实 RIR  train/test: 48199/12049

清单已写入 /content/drive/MyDrive/Audio AI/RTSE/manifest.json


## 4. 合成固定测试集（中文 WenetSpeech + DNS 噪声 + 双 RIR）

**测试集必须固定下来**（预先合成好存盘），不能在线随机生成 ——
否则每次评测的噪声段和 SNR 都不同，前后两次跑出来的 CER 没有可比性。
训练集则相反，必须在线随机混音以扩大有效数据量。

### 分层设计

| 维度 | 取值 | 用途 |
|---|---|---|
| SNR | −5 / 0 / 5 / 10 / 15 dB | −5~0 压力测试；5~10 典型远场办公；15 验证"无害性" |
| 噪声 | 稳态 / 非稳态 | DSP vs NN 的核心对照 |
| RIR | 合成（RT60 0.2/0.4/0.6/0.8）+ 真实 | 受控扫描 + 泛化检验 |

**SNR 阶梯的理由**：
- **−5 ~ 0 dB**：重度淹没，最容易暴露"过抑制"——听着安静了但 ASR 断崖式下跌；
- **5 ~ 10 dB**：日常办公/会议室远场的典型信噪比，工程落地价值最核心的区间；
- **15 dB**：轻噪场景，评估**无害性**——好的增强算法在信噪比已经不错时
  不该引入额外失真让 CER 反而变差。

**RT60 阶梯的理由**：0.2 s 小型吸音房间；0.4~0.5 s 标准会议室/办公室；
0.8 s 大型未做吸音的教室或高挑大堂。混响会造成时域拖尾（前一个音素的反射
覆盖后一个），破坏 ASR 的声学特征边界。

In [10]:
import io
import numpy as np
import soundfile as sf
from rtse.audio.io import write_audio
from rtse.data.synth import mix_at_snr, apply_rir, make_rir, speech_active_mask
from rtse.dsp.rt60 import estimate_t60

SNRS = [-5, 0, 5, 10, 15]
RT60S = [0.2, 0.4, 0.6, 0.8]
PER_CELL = 3 if SMOKE_RUN else 15
MIN_SEG_SEC, MAX_SEG_SEC = 3.0, 15.0

def decode_wns(rec):
    """从 parquet 一行里取出 (16kHz 波形, 中文转写)。

    HF 的音频列是 {'bytes': ..., 'path': ...}，用 soundfile 从内存解码
    （opus in ogg，libsndfile 1.2+ 支持），再重采样到项目统一的 16 kHz。
    """
    a = rec['audio']
    data, sr = sf.read(io.BytesIO(a['bytes']), dtype='float64', always_2d=False)
    if data.ndim == 2:
        data = data.mean(axis=1)
    if sr != 16000:
        import soxr
        data = soxr.resample(data, sr, 16000, quality='VHQ')
    txt = rec.get('text') or rec.get('sentence') or rec.get('transcription') or ''
    return data, str(txt)

# 挑长度合适、有转写的样本
cands = []
for i in range(len(wns)):
    r = wns.iloc[i]
    try:
        y, t = decode_wns(r)
    except Exception:
        continue
    d = y.size / 16000
    if MIN_SEG_SEC <= d <= MAX_SEG_SEC and len(t.strip()) >= 5:
        cands.append((y, t.strip()))
    if len(cands) >= (120 if SMOKE_RUN else 400):
        break
print(f'可用 WenetSpeech 样本 {len(cands)} 条')
assert cands, 'WenetSpeech 没解出可用样本，检查 parquet 的列名'
print('样例转写:', cands[0][1][:40])
print('样例时长: %.2f 秒' % (cands[0][0].size/16000))

可用 WenetSpeech 样本 400 条
样例转写: 好首先说一下刚才这个经理说完的这个销售问题咱再说一下咱们的商场问题首先咱们商场上
样例时长: 12.37 秒


In [11]:
# 实验格：SNR × 噪声平稳性 × RIR 条件
rir_conditions = [('synth', t) for t in RT60S] + [('real', None)]
cells = [{'snr': s, 'noise_kind': nk, 'rir_kind': rk, 'rt60': rt}
         for s in SNRS for nk in ['stationary', 'nonstationary']
         for rk, rt in rir_conditions]
print(f'{len(cells)} 格 × {PER_CELL} 条 = {len(cells)*PER_CELL} 个样本')
print(f'  SNR {len(SNRS)} × 噪声 2 × RIR {len(rir_conditions)}（合成 {len(RT60S)} 档 + 真实 1 档）')

os.makedirs(f'{TESTSET_DIR}/audio', exist_ok=True)
# 先清空，保证目录内容严格等于 index.json —— 改小 PER_CELL 重跑时，
# 上一轮遗留的文件不会被覆盖，会变成索引里没有的孤儿文件混进 zip。
shutil.rmtree(f'{TESTSET_DIR}/audio', ignore_errors=True)
os.makedirs(f'{TESTSET_DIR}/audio', exist_ok=True)

rng = np.random.default_rng(20260807)
noise_by_kind = {'stationary': st_test, 'nonstationary': ns_test}
records, sidx = [], 0

def take_noise(paths, n, rng_):
    """取一段有真实交流能量的噪声；常量/静音局部片段自动重试（I-26）。"""
    for _ in range(100):
        y = read_audio(paths[rng_.integers(len(paths))])
        if y.size < 8000 or not np.all(np.isfinite(y)):
            continue
        if y.size < n:
            y = np.tile(y, int(np.ceil(n / y.size)))
        s = rng_.integers(0, y.size - n + 1) if y.size > n else 0
        segment = y[s:s+n]
        segment_ac = segment - segment.mean()
        if segment.size == n and np.mean(segment_ac ** 2) > 1e-12:
            return segment
    raise RuntimeError(
        f'连续 100 次没有取到有效噪声片段（候选文件 {len(paths)} 个）。'
        '噪声池里可能混入了大量静音/常量文件，请检查分类阶段的 skipped 数量。')

for ci, cell in enumerate(tqdm(cells, desc='合成测试集')):
    for k in range(PER_CELL):
        clean, text = cands[(ci * PER_CELL + k) % len(cands)]
        n = clean.size
        clean = clean / (np.max(np.abs(clean)) + 1e-9) * 0.7

        if cell['rir_kind'] == 'synth':
            rir = make_rir(cell['rt60'], rng=rng)
            rt60_actual = round(float(estimate_t60(rir)), 3)
        else:
            rir = read_audio(rir_test[rng.integers(len(rir_test))])
            rt60_actual = round(float(estimate_t60(rir)), 3)
        wet = apply_rir(clean, rir)

        noise = take_noise(noise_by_kind[cell['noise_kind']], n, rng)
        noisy, scaled_noise = mix_at_snr(wet, noise, cell['snr'], rng=rng)
        # 实测 SNR：和 rt60 一样，**标称是请求，实测才是事实**。
        # I-22 之后给混响加了 measured 字段，但 SNR 一直只记标称值，
        # I-24（直流把功率统计撑歪）正是从这个缺口漏过去的。
        # 口径必须和 mix_at_snr 内部一致：去直流 + 只在语音活跃段上算。
        _s = wet - wet.mean()
        _n = scaled_noise - scaled_noise.mean()
        _m = speech_active_mask(_s)
        _pn = float(np.mean(_n ** 2))
        assert np.isfinite(_pn) and _pn > 0, '缩放噪声没有交流能量，不能生成 SNR 样本'
        snr_actual = round(float(10 * np.log10(np.mean(_s[_m] ** 2) / _pn)), 2)
        assert np.isfinite(snr_actual), f'SNR 实测异常: {snr_actual}'

        stem = f"{sidx:05d}_{cell['noise_kind']}_snr{cell['snr']}_{cell['rir_kind']}"
        write_audio(f'{TESTSET_DIR}/audio/{stem}_noisy.wav', noisy)
        # 参考 = **混响后**的干净语音，不是原始干信号。否则降噪模型会因为
        # "没能去掉混响"被扣分，把降噪和去混响两件事混在一起。
        write_audio(f'{TESTSET_DIR}/audio/{stem}_clean.wav', wet)
        records.append({
            'id': stem, 'noisy': f'audio/{stem}_noisy.wav', 'clean': f'audio/{stem}_clean.wav',
            'text': text, 'duration_s': round(n/16000, 2),
            'snr': cell['snr'], 'snr_measured': snr_actual,
            'noise_kind': cell['noise_kind'],
            'rir_kind': cell['rir_kind'],
            'rt60_nominal': cell['rt60'], 'rt60_measured': rt60_actual,
        })
        sidx += 1

Path(f'{TESTSET_DIR}/index.json').write_text(json.dumps({
    'sample_rate': 16000, 'version': 'dns_wenetspeech', 'per_cell': PER_CELL,
    # **让数据自己声明限制**，不靠使用者记得（见 docs/ISSUES.md I-30）。
    # WenetSpeech test_meeting 是真实会议室远场录音，实测段内 SNR 仅 3.7~12.4 dB，
    # 本身含可观背景噪声与混响，**不满足有参考指标对"干净参考"的要求**：
    # 增强器把这些固有噪声去掉是对的，却会因此偏离参考而被判低分。
    # rtse-eval 读到这个标志会自动跳过 SI-SDR/STOI/ESTOI 并说明原因。
    # CER 不受影响——它比的是识别文本与人工转写，不经过参考音频。
    'reference_is_clean': False,
    'reference_note': ('WenetSpeech test_meeting 是真实会议室远场录音，本身含噪与混响，'
                       '不满足有参考指标对"干净参考"的要求；CER 不受影响。见 ISSUES.md I-30。'),
    'snrs': SNRS, 'rt60s': RT60S, 'records': records,
}, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'\n测试集 {len(records)} 个样本 → {TESTSET_DIR}')
!du -sh "{TESTSET_DIR}"

50 格 × 15 条 = 750 个样本
  SNR 5 × 噪声 2 × RIR 5（合成 4 档 + 真实 1 档）


合成测试集:   0%|          | 0/50 [00:00<?, ?it/s]


测试集 750 个样本 → /content/drive/MyDrive/Audio AI/RTSE/testset
313M	/content/drive/MyDrive/Audio AI/RTSE/testset


### 校验

两件事必须确认，否则后面算出来的 CER 不可信：

1. **文本与音频对得上**：WenetSpeech 的音频是完整句子，没有做任何截断
   （`MIN_SEG_SEC`/`MAX_SEG_SEC` 是**筛选**条件，不是裁剪）——
   这一点靠代码结构保证，这里断言一下。
2. **合成 RIR 的实际 RT60 与标称值相符**：`make_rir` 的镜像阶数曾经写死，
   导致 RT60 超过 0.6 s 后完全失效（见 `docs/ISSUES.md` I-22）。
   已修复，但每次生成数据都值得复核一遍。

In [12]:
import statistics as st

# 不依赖内存里的 records：Colab 重启、跳着跑 cell，或某个生成 cell 中途失败后，
# records 可能是空列表，但磁盘上的 index.json 仍然完整（I-26）。
index_path = Path(f'{TESTSET_DIR}/index.json')
assert index_path.exists(), f'找不到 {index_path}，请先运行测试集生成 cell'
index_data = json.loads(index_path.read_text(encoding='utf-8'))
records = index_data.get('records', [])
assert records, f'{index_path} 中 records 为空，请重新运行测试集生成 cell'
SNRS = index_data.get('snrs', sorted({r['snr'] for r in records}))
RT60S = index_data.get('rt60s', sorted({r['rt60_nominal'] for r in records
                                         if r.get('rt60_nominal') is not None}))
max_seg_sec = globals().get('MAX_SEG_SEC', 18.0)

# 1) 音频没有被截断：所有时长都在筛选区间内，且不存在"卡在上限"的聚集
durs = [r['duration_s'] for r in records]
at_max = sum(1 for d in durs if abs(d - max_seg_sec) < 0.02)
print(f'时长 {min(durs):.2f}~{max(durs):.2f} 秒，均值 {st.mean(durs):.2f}')
print(f'恰好卡在上限 {max_seg_sec}s 的样本: {at_max} 条（大量聚集才说明被裁过）')
assert at_max < len(records) * 0.05, '大量样本卡在时长上限，检查是否误加了裁剪'

# 2) SNR 标称 vs 实测 —— 这一条能在生成时就拦住 I-24 那类问题
print('\nSNR 标称 vs 实测:')
bad_snr = [r for r in records if not np.isfinite(r.get('snr_measured', np.nan))]
assert not bad_snr, (
    f'有 {len(bad_snr)} 条 SNR 为 inf/nan：{[r["id"] for r in bad_snr[:10]]}。'
    ' 这些样本选到了静音/常量噪声，请用修复后的生成 cell 重新生成。')
snr_bias = {}
for s in SNRS:
    ms = [r['snr_measured'] for r in records if r['snr'] == s]
    if ms:
        snr_bias[s] = st.mean(ms) - s
        print(f'  标称 {s:>3} dB → 实测均值 {st.mean(ms):6.2f} dB（偏差 {snr_bias[s]:+.2f}）')
worst = max(snr_bias.items(), key=lambda kv: abs(kv[1]), default=(None, 0.0))
assert abs(worst[1]) < 1.0, (
    f'标称 {worst[0]} dB 那一档的实测 SNR 偏了 {worst[1]:+.2f} dB。'
    ' 混音的功率统计口径有问题（直流？活跃段掩码？），见 docs/ISSUES.md I-24')
# 不处理的信号，其真实 SNR 必须随标称单调上升——这是一条**与合成代码无关**的外部
# 不变量。I-24 当初就是靠它露的马脚：汇总均值完全看不出问题，按 SNR 排开才暴露。
means = [st.mean([r['snr_measured'] for r in records if r['snr'] == s]) for s in sorted(SNRS)]
assert all(b > a for a, b in zip(means, means[1:])), (
    f'实测 SNR 没有随标称单调上升：{[round(m, 2) for m in means]}，见 docs/ISSUES.md I-24')
print('  ✓ 实测 SNR 随标称单调上升')

# 3) 合成 RIR 的实际 RT60 vs 标称
print('\n合成 RIR 的 RT60 标称 vs 实测:')
for t in RT60S:
    ms = [r['rt60_measured'] for r in records if r['rir_kind']=='synth' and r['rt60_nominal']==t]
    if ms:
        print(f'  标称 {t}s → 实测均值 {st.mean(ms):.3f}s')
real_rt = [r['rt60_measured'] for r in records if r['rir_kind']=='real'
           and r['rt60_measured'] == r['rt60_measured']]
if real_rt:
    print(f'\n真实 RIR 实测 RT60: {min(real_rt):.2f}~{max(real_rt):.2f}s，'
          f'中位数 {st.median(real_rt):.2f}s')

# 3) 分层是否齐全
import collections
print('\n各层样本数:', dict(collections.Counter(
    (r['noise_kind'], r['rir_kind']) for r in records)))
print('\n校验通过。')

时长 3.03~14.91 秒，均值 6.81
恰好卡在上限 15.0s 的样本: 0 条（大量聚集才说明被裁过）

SNR 标称 vs 实测:
  标称  -5 dB → 实测均值  -5.00 dB（偏差 +0.00）
  标称   0 dB → 实测均值   0.00 dB（偏差 +0.00）
  标称   5 dB → 实测均值   5.00 dB（偏差 +0.00）
  标称  10 dB → 实测均值  10.00 dB（偏差 +0.00）
  标称  15 dB → 实测均值  15.00 dB（偏差 +0.00）
  ✓ 实测 SNR 随标称单调上升

合成 RIR 的 RT60 标称 vs 实测:
  标称 0.2s → 实测均值 0.194s
  标称 0.4s → 实测均值 0.476s
  标称 0.6s → 实测均值 0.709s
  标称 0.8s → 实测均值 0.910s

真实 RIR 实测 RT60: 0.09~5.28s，中位数 2.20s

各层样本数: {('stationary', 'synth'): 300, ('stationary', 'real'): 75, ('nonstationary', 'synth'): 300, ('nonstationary', 'real'): 75}

校验通过。


## 5. 打包测试集，下载到本地

In [13]:
!cd "{DRIVE}" && rm -f testset.zip && zip -q -r testset.zip testset && ls -lh testset.zip
print()
print('下一步：')
print('  1. 继续跑 02_train.ipynb（数据清单已就绪）')
print(f'  2. 从 Drive 下载 {DRIVE}/testset.zip，解压到本地项目的 data/ 下，')
print('     使 data/testset/index.json 存在，然后本地 `uv run rtse-eval` 就能跑')

-rw------- 1 root root 269M Aug 17 13:52 testset.zip

下一步：
  1. 继续跑 02_train.ipynb（数据清单已就绪）
  2. 从 Drive 下载 /content/drive/MyDrive/Audio AI/RTSE/testset.zip，解压到本地项目的 data/ 下，
     使 data/testset/index.json 存在，然后本地 `uv run rtse-eval` 就能跑
